# Dependency setup
+ Pre-clean - none 
+ Purchase - none
+ Mortar - Purchase, Pre-clean; both Succeeded
+ Tile - Mortar Started, Running, Succeeded, Complete
+ Grout - Tile Succeeded
+ Post-clean - Grout Succeeded, Fail, Complete, Paused, Cancelled OR Tile Cancelled

In [0]:
%sql
with p as (
  select 'Pre-clean' as name, 'Clean subfloor and prepare surface.' as description
  union 
  select 'Purchase', 'Choose and purchase tile, mortar, grout & other materials.'
  union 
  select 'Mortar', 'Mix and spread mortar.'
  union 
  select 'Tile', 'Set tile; cut to fit as needed.'
  union 
  select 'Grout', 'Apply grout and seal as needed.'
  union 
  select 'Post-clean', 'Clean up and finish.'
)
insert into process (name, description)
select p.name, p.description
from p
where not exists (select * from process where name = p.name)

In [0]:
%sql
-- delete from process_dependency;

with pd as (
  -- Mortar - accept group_id = 0 as all these can AND together
  select  p.id as process_id, 0 as group_id, a.id as antecedent_id
      ,   collect_set(s.id) as state_ids
  from process p 
  join process a on 1 = 1
  left join execution_state s on 1 = 1
  where p.name = 'Mortar' 
    and a.name in ('Pre-clean', 'Purchase') 
    and s.name in ('Succeeded', 'Complete')
  group by p.id, a.id
  union
  -- Tile - accept group_id = 0 as there's only one antecedent
  select  p.id as process_id, 0 as group_id, a.id as antecedent_id
      ,   collect_set(s.id) as state_ids
  from process p 
  join process a on 1 = 1
  left join execution_state s on 1 = 1
  where p.name = 'Tile' 
    and a.name in ('Mortar') 
    and s.name in ('Running', 'Resumed', 'Succeeded', 'Complete')
  group by p.id, a.id
  union
  -- Grout - accept group_id = 0 as there's only one antecedent
  select  p.id as process_id, 0 as group_id, a.id as antecedent_id
      ,   collect_set(s.id) as state_ids
  from process p 
  join process a on 1 = 1
  left join execution_state s on 1 = 1
  where p.name = 'Grout' 
    and a.name in ('Tile') 
    and s.name in ('Succeeded')
  group by p.id, a.id
  union
  -- Post-clean - Use group_id here as we should clean up under several conditions
  select  p.id as process_id, 0 as group_id, a.id as antecedent_id
      ,   collect_set(s.id) as state_ids
  from process p 
  join process a on 1 = 1
  left join execution_state s on 1 = 1
  where p.name = 'Post-clean' 
    and a.name in ('Grout')
    and s.name in ('Paused', 'Cancelled', 'Succeeded', 'Failed', 'Complete')
  group by p.id, a.id
  union
  select  p.id as process_id, 1 as group_id, a.id as antecedent_id
      ,   collect_set(s.id) as state_ids
  from process p 
  join process a on 1 = 1
  left join execution_state s on 1 = 1
  where p.name = 'Post-clean' 
    and a.name in ('Tile')
    and s.name in ('Paused', 'Cancelled', 'Succeeded', 'Failed', 'Complete')
  group by p.id, a.id
  union
  select  p.id as process_id, 2 as group_id, a.id as antecedent_id
      ,   collect_set(s.id) as state_ids
  from process p 
  join process a on 1 = 1
  left join execution_state s on 1 = 1
  where p.name = 'Post-clean' 
    and a.name in ('Mortar')
    and s.name in ('Paused', 'Cancelled', 'Succeeded', 'Failed', 'Complete')
  group by p.id, a.id
)
insert into process_dependency (process_id, group_id, antecedent_id, state_ids)
select    pd.process_id, pd.group_id, pd.antecedent_id, pd.state_ids
from pd
where not exists (select * from process_dependency where process_id = pd.process_id and antecedent_id = pd.antecedent_id)

## Show Dependencies

In [0]:
%sql
with ord as (
    -- Dummy up the order of the processes for display purposes
            select  'Pre-clean' as process_name, 0 as ord
    union   select  'Purchase', 0
    union   select  'Mortar', 1
    union   select  'Tile', 2
    union   select  'Grout', 3
    union   select  'Post-clean', 4
), pda as (
    -- Get names of processes and their antecedents, along with the acceptable states
    select      p.name as process_name, pd.group_id, coalesce(a.name, '-- No dependency') as antecedent_name
        ,       explode(coalesce(pd.state_ids, array(0))) as state_id
    from        process p
    left join   process_dependency pd on pd.process_id = p.id
    left join   process a on a.id = pd.antecedent_id
)

-- Put all this together and collect a list of state names for each condition
select      ord.ord
       ,    pda.process_name, pda.group_id, pda.antecedent_name
       ,    collect_list(s.name) as state_names
from        pda
join        ord on ord.process_name = pda.process_name
left join   execution_state s on s.id = pda.state_id
group by    ord.ord, pda.process_name, pda.group_id, pda.antecedent_name
order by    ord.ord, pda.process_name, pda.group_id, pda.antecedent_name
